In [ ]:
import pandas as pd
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
df=pd.read_csv('../data/cleaned_dataset.csv')
X = df["comment"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def normalize_farsi(text):
    text = text.lower()
    text = text.replace("ي", "ی")
    text = text.replace("ك", "ک")
    text = text.replace("ـ", "")
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
def tokenize(text):
    return normalize_farsi(text).split()

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = pd.Series(y_train).reset_index(drop=True)
y_test = pd.Series(y_test).reset_index(drop=True)
    
counter = Counter()

for text in X_train:
    counter.update(tokenize(text))

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, _ in counter.most_common(30000):
    vocab[word] = len(vocab)
print("fff")

MAX_LEN = 120

def encode(text):
    return [vocab.get(w, 1) for w in tokenize(text)]

def pad(seq):
    return seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(seq))

class ToxicDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.values
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = pad(encode(self.texts[idx]))
        y = self.labels[idx]

        return torch.tensor(x), torch.tensor(y, dtype=torch.float)


train_dataset = ToxicDataset(X_train, y_train)
test_dataset = ToxicDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        return self.fc(hidden).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
epochs = 1

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")

Epoch 1/1 | Loss: 57.2211


In [16]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

Accuracy: 0.8489312536106297
              precision    recall  f1-score   support

         0.0       0.88      0.90      0.89      2383
         1.0       0.77      0.74      0.75      1079

    accuracy                           0.85      3462
   macro avg       0.83      0.82      0.82      3462
weighted avg       0.85      0.85      0.85      3462

